In [4]:
import tensorflow.lite as tflite
import tensorflow as tf
import cv2
import numpy as np

# Carregar o modelo TFLite
interpretador = tflite.Interpreter(model_path='./Modelos/modelo_novo.tflite')
interpretador.allocate_tensors()

input_details = interpretador.get_input_details()
output_details = interpretador.get_output_details()

# Iniciar captura de vídeo
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao abrir a câmera.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        print("Falha ao capturar imagem.")
        break

    # Redimensionar a imagem e normalizar
    img = cv2.resize(frame, (128, 128))  # Ajuste de acordo com o seu modelo
    img = np.expand_dims(img, axis=0).astype(np.float32) / 255.0

    # Definir o tensor de entrada
    interpretador.set_tensor(input_details[0]['index'], img)
    interpretador.invoke()

    # Obter a saída do modelo
    output = interpretador.get_tensor(output_details[0]['index'])
    label = "Gato" if output[0][0] < 0.5 else "Cachorro"

    # Exibir o rótulo na imagem
    cv2.putText(frame, label, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Exibir a imagem com a previsão
    cv2.imshow("Detecção", frame)

    # Fechar a janela com a tecla 'Esc'
    if cv2.waitKey(1) & 0xFF == 27:  # 27 é o código ASCII para 'Esc'
        break

cap.release()
cv2.destroyAllWindows()
